In [1]:
import ee
from pathlib import PurePosixPath

ee.Authenticate(
    auth_mode='notebook',  # or 'gcloud' or 'appdefault' in other contexts
    scopes=['https://www.googleapis.com/auth/earthengine',
            'https://www.googleapis.com/auth/devstorage.read_write',
            'https://www.googleapis.com/auth/drive']
    #, quiet=False  # force re-authentication even if cached tokens exist
)
ee.Initialize(project="ee-tymc5571-cbi-module")


c:\Users\tymc5571\AppData\Local\miniconda3\envs\cbi-module\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [3]:
collection_id = "projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev_4326"

# 1. List all assets in the collection
assets = ee.data.listAssets({'parent': collection_id})

# 2. Delete each image
for asset in assets.get('assets', []):
    print(f"Deleting image: {asset['name']}")
    ee.data.deleteAsset(asset['name'])

# 3. Delete the collection itself
print(f"Deleting collection: {collection_id}")
ee.data.deleteAsset(collection_id)

Deleting image: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev_4326/WUMI2024a_allsev_1985
Deleting image: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev_4326/WUMI2024a_allsev_1986
Deleting image: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev_4326/WUMI2024a_allsev_1987
Deleting image: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev_4326/WUMI2024a_allsev_1988
Deleting image: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev_4326/WUMI2024a_allsev_1989
Deleting image: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev_4326/WUMI2024a_allsev_1990
Deleting image: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev_4326/WUMI2024a_allsev_1991
Deleting image: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev_4326/WUMI2024a_allsev_1992
Deleting image: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev_4326/WUMI2024a_allsev_1993
Deleting image: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev_4326/WUMI2024a_allsev_1994
Deleting i

In [ ]:

BASE_DIR = "projects/ee-tymc5571-cbi-module/assets"

COLLECTION_NAMES = [
    "bccbi_firedv2_west",
    "bccbi_welty_west",
    "bccbi_wumi_west_circles",
    "bccbi_wumi_west_nocircles",
    "west_wall_to_wall_bccbi",
    "WUMI2024a_allsev",
]

DRY_RUN = True
OVERWRITE = False


def basename(asset_id: str) -> str:
    return PurePosixPath(asset_id).name


def list_assets(parent: str) -> list[dict]:
    result = ee.data.listAssets({"parent": parent})
    return result.get("assets", [])


def get_asset_type(asset: dict) -> str | None:
    # Earth Engine returns types like IMAGE, IMAGE_COLLECTION, FOLDER
    # depending on endpoint/client version.
    return asset.get("type")


def ensure_collection(collection_id: str) -> None:
    try:
        ee.data.getAsset(collection_id)
        print(f"Collection exists: {collection_id}")
    except Exception:
        ee.data.createAsset({"type": "IMAGE_COLLECTION"}, collection_id)
        print(f"Created collection: {collection_id}")


def build_collection_map(base_dir: str, collection_names: list[str]) -> dict[str, str]:
    return {name: f"{base_dir}/{name}" for name in collection_names}


def find_matching_images(
    assets: list[dict],
    collection_names: list[str],
    base_dir: str,
) -> dict[str, list[str]]:
    matches = {name: [] for name in collection_names}

    collection_ids = {f"{base_dir}/{name}" for name in collection_names}

    for asset in assets:
        asset_id = asset["name"]
        asset_type = get_asset_type(asset)
        asset_name = basename(asset_id)

        # Only move IMAGE assets
        if asset_type != "IMAGE":
            continue

        # Skip anything already inside one of the target collections
        if any(asset_id.startswith(collection_id + "/") for collection_id in collection_ids):
            continue

        for collection_name in collection_names:
            if asset_name.startswith(collection_name):
                matches[collection_name].append(asset_id)
                break

    return matches


def move_assets_to_collections(
    matches: dict[str, list[str]],
    collection_map: dict[str, str],
    dry_run: bool = True,
    overwrite: bool = False,
) -> None:
    for collection_name, source_ids in matches.items():
        collection_id = collection_map[collection_name]
        ensure_collection(collection_id)

        print("\n" + "=" * 80)
        print(f"Collection: {collection_id}")
        print(f"Matched images: {len(source_ids)}")

        for src in sorted(source_ids):
            dst = f"{collection_id}/{basename(src)}"
            print(f"{'DRY RUN' if dry_run else 'MOVE'}: {src} -> {dst}")

            if dry_run:
                continue

            if overwrite:
                try:
                    ee.data.deleteAsset(dst)
                    print(f"Deleted existing destination: {dst}")
                except Exception:
                    pass

            ee.data.renameAsset(src, dst)


def main():
    assets = list_assets(BASE_DIR)
    collection_map = build_collection_map(BASE_DIR, COLLECTION_NAMES)
    matches = find_matching_images(assets, COLLECTION_NAMES, BASE_DIR)

    print("Summary:")
    for collection_name, source_ids in matches.items():
        print(f"  {collection_name}: {len(source_ids)} image(s)")

    move_assets_to_collections(
        matches=matches,
        collection_map=collection_map,
        dry_run=DRY_RUN,
        overwrite=OVERWRITE,
    )


if __name__ == "__main__":
    main()

Summary:
  bccbi_firedv2_west: 0 image(s)
  bccbi_welty_west: 0 image(s)
  bccbi_wumi_west_circles: 0 image(s)
  bccbi_wumi_west_nocircles: 0 image(s)
  west_wall_to_wall_bccbi: 0 image(s)
  WUMI2024a_allsev: 0 image(s)
Collection exists: projects/ee-tymc5571-cbi-module/assets/bccbi_firedv2_west

Collection: projects/ee-tymc5571-cbi-module/assets/bccbi_firedv2_west
Matched images: 0
Collection exists: projects/ee-tymc5571-cbi-module/assets/bccbi_welty_west

Collection: projects/ee-tymc5571-cbi-module/assets/bccbi_welty_west
Matched images: 0
Collection exists: projects/ee-tymc5571-cbi-module/assets/bccbi_wumi_west_circles

Collection: projects/ee-tymc5571-cbi-module/assets/bccbi_wumi_west_circles
Matched images: 0
Collection exists: projects/ee-tymc5571-cbi-module/assets/bccbi_wumi_west_nocircles

Collection: projects/ee-tymc5571-cbi-module/assets/bccbi_wumi_west_nocircles
Matched images: 0
Collection exists: projects/ee-tymc5571-cbi-module/assets/west_wall_to_wall_bccbi

Collection: p

In [6]:
import ee
import json
from collections import Counter


BASE = "projects/ee-tymc5571-cbi-module/assets"

COLLECTIONS = [
    "bccbi_firedv2_west",
    "bccbi_welty_west",
    "bccbi_wumi_west_circles",
    "bccbi_wumi_west_nocircles",
    "west_wall_to_wall_bccbi",
    "WUMI2024a_allsev",
]


def list_images_in_collection(collection_id: str) -> list[dict]:
    out = ee.data.listImages({"parent": collection_id})
    return out.get("images", [])


def get_band_projection_info(image_id: str, band_name: str | None = None) -> dict:
    info = ee.Image(image_id).getInfo()

    bands = info.get("bands", [])
    if not bands:
        return {
            "image_id": image_id,
            "band_used": None,
            "crs": None,
            "crs_transform": None,
            "dimensions": None,
            "note": "No bands found"
        }

    if band_name is None:
        band = bands[0]
    else:
        matching = [b for b in bands if b.get("id") == band_name]
        if not matching:
            raise ValueError(f"Band '{band_name}' not found in {image_id}")
        band = matching[0]

    return {
        "image_id": image_id,
        "band_used": band.get("id"),
        "crs": band.get("crs"),
        "crs_transform": band.get("crs_transform"),
        "dimensions": band.get("dimensions"),
    }


def summarize_collection(collection_id: str, band_name: str | None = None) -> dict:
    images = list_images_in_collection(collection_id)

    records = []
    for img in images:
        image_id = img["name"]
        rec = get_band_projection_info(image_id, band_name=band_name)
        records.append(rec)

    crs_counts = Counter(r["crs"] for r in records)
    transform_counts = Counter(tuple(r["crs_transform"]) if r["crs_transform"] else None for r in records)

    return {
        "collection_id": collection_id,
        "n_images": len(records),
        "unique_crs": len(crs_counts),
        "unique_transforms": len(transform_counts),
        "crs_counts": dict(crs_counts),
        "transform_counts": {str(k): v for k, v in transform_counts.items()},
        "records": records,
    }


def main():
    all_results = []

    for name in COLLECTIONS:
        collection_id = f"{BASE}/{name}"
        print("=" * 100)
        print(collection_id)

        summary = summarize_collection(collection_id)
        all_results.append(summary)

        print(f"Images: {summary['n_images']}")
        print(f"Unique CRS: {summary['unique_crs']}")
        print(f"Unique transforms: {summary['unique_transforms']}")
        print("CRS counts:")
        for crs, n in summary["crs_counts"].items():
            print(f"  {crs}: {n}")

        if summary["n_images"] > 0:
            print("Example image:")
            print(json.dumps(summary["records"][0], indent=2))

    with open("gee_collection_projection_check.json", "w") as f:
        json.dump(all_results, f, indent=2)

    print("\nSaved full report to gee_collection_projection_check.json")


if __name__ == "__main__":
    main()

projects/ee-tymc5571-cbi-module/assets/bccbi_firedv2_west
Images: 25
Unique CRS: 1
Unique transforms: 1
CRS counts:
  EPSG:5070: 25
Example image:
{
  "image_id": "projects/ee-tymc5571-cbi-module/assets/bccbi_firedv2_west/bccbi_firedv2_west_2000",
  "band_used": "CBI_bc",
  "crs": "EPSG:5070",
  "crs_transform": [
    30,
    0,
    -2361600,
    0,
    -30,
    3177450
  ],
  "dimensions": [
    61900,
    72855
  ]
}
projects/ee-tymc5571-cbi-module/assets/bccbi_welty_west
Images: 36
Unique CRS: 1
Unique transforms: 1
CRS counts:
  EPSG:5070: 36
Example image:
{
  "image_id": "projects/ee-tymc5571-cbi-module/assets/bccbi_welty_west/bccbi_welty_west_1985",
  "band_used": "CBI_bc",
  "crs": "EPSG:5070",
  "crs_transform": [
    30,
    0,
    -2361600,
    0,
    -30,
    3177450
  ],
  "dimensions": [
    61900,
    72855
  ]
}
projects/ee-tymc5571-cbi-module/assets/bccbi_wumi_west_circles
Images: 40
Unique CRS: 1
Unique transforms: 1
CRS counts:
  EPSG:5070: 40
Example image:
{
  "ima

In [16]:
import ee
import re


SRC_COLLECTION = "projects/ee-tymc5571-cbi-module/assets/forest_masks"
DST_COLLECTION = "projects/ee-tymc5571-cbi-module/assets/forest_masks_og"

DRY_RUN = False


def list_images(collection_id: str) -> list[str]:
    result = ee.data.listImages({"parent": collection_id})
    return [img["name"] for img in result.get("images", [])]


def basename(asset_id: str) -> str:
    return asset_id.rstrip("/").split("/")[-1]


def move_images(src_collection: str, dst_collection: str, dry_run: bool = True) -> None:
    image_ids = list_images(src_collection)

    print(f"Found {len(image_ids)} images in {src_collection}")

    for src_id in sorted(image_ids):
        img_name = basename(src_id)
        dst_id = f"{dst_collection}/{img_name}"

        print(f"{'DRY RUN' if dry_run else 'MOVE'}: {src_id} -> {dst_id}")

        if not dry_run:
            ee.data.renameAsset(src_id, dst_id)

    print("Done.")


move_images(SRC_COLLECTION, DST_COLLECTION, dry_run=DRY_RUN)

Found 40 images in projects/ee-tymc5571-cbi-module/assets/forest_masks
MOVE: projects/ee-tymc5571-cbi-module/assets/forest_masks/year_1985 -> projects/ee-tymc5571-cbi-module/assets/forest_masks_og/year_1985
MOVE: projects/ee-tymc5571-cbi-module/assets/forest_masks/year_1986 -> projects/ee-tymc5571-cbi-module/assets/forest_masks_og/year_1986
MOVE: projects/ee-tymc5571-cbi-module/assets/forest_masks/year_1987 -> projects/ee-tymc5571-cbi-module/assets/forest_masks_og/year_1987
MOVE: projects/ee-tymc5571-cbi-module/assets/forest_masks/year_1988 -> projects/ee-tymc5571-cbi-module/assets/forest_masks_og/year_1988
MOVE: projects/ee-tymc5571-cbi-module/assets/forest_masks/year_1989 -> projects/ee-tymc5571-cbi-module/assets/forest_masks_og/year_1989
MOVE: projects/ee-tymc5571-cbi-module/assets/forest_masks/year_1990 -> projects/ee-tymc5571-cbi-module/assets/forest_masks_og/year_1990
MOVE: projects/ee-tymc5571-cbi-module/assets/forest_masks/year_1991 -> projects/ee-tymc5571-cbi-module/assets/for

# Re-project WUMI

In [14]:
import ee
import re

ee.Initialize(project="ee-tymc5571-cbi-module")

SRC_COLLECTION = "projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev_4326"
DST_COLLECTION = "projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev"

TARGET_CRS = "EPSG:5070"
TARGET_TRANSFORM = [30, 0, -2361600, 0, -30, 3177450]

MAX_PIXELS = 1e13
SHARD_SIZE = 512

START_TASKS = True
TEST_1 = True


def list_images(collection_id):
    result = ee.data.listImages(collection_id)
    return [img["name"] for img in result.get("images", [])]


def basename(asset_id):
    return asset_id.rstrip("/").split("/")[-1]


def safe_description(name):
    return re.sub(r"[^A-Za-z0-9_-]", "_", name)[:100]


def asset_exists(asset_id):
    try:
        ee.data.getAsset(asset_id)
        return True
    except Exception:
        return False


def launch_export(image_id):
    name = basename(image_id)
    dst_id = f"{DST_COLLECTION}/{name}"

    if asset_exists(dst_id):
        print(f"Skipping existing asset: {dst_id}")
        return None

    image = ee.Image(image_id)

    task = ee.batch.Export.image.toAsset(
        image=image,
        description=safe_description(f"to5070_{name}"),
        assetId=dst_id,
        region=image.geometry(),
        crs=TARGET_CRS,
        crsTransform=TARGET_TRANSFORM,
        maxPixels=MAX_PIXELS,
        shardSize=SHARD_SIZE
    )

    if START_TASKS:
        task.start()

    print(f"{'Started' if START_TASKS else 'Prepared'}: {dst_id}")
    return task


def main():
    image_ids = sorted(list_images(SRC_COLLECTION))
    print(f"Found {len(image_ids)} images")

    if TEST_1:
        image_ids = image_ids[:1]
        print("TEST_1=True -> running only first image")

    tasks = []
    for image_id in image_ids:
        task = launch_export(image_id)
        if task is not None:
            tasks.append(task)

    print(f"\nTasks created: {len(tasks)}")


if __name__ == "__main__":
    main()

Found 40 images
TEST_1=True -> running only first image
Started: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev/WUMI2024a_allsev_1985

Tasks created: 1


# New forest masks

In [30]:
import ee
ee.Initialize()

SRC_COL = "projects/ee-tymc5571-cbi-module/assets/forest_masks_og"

img = ee.ImageCollection(SRC_COL).first()

info = img.getInfo()

band0 = info["bands"][0]

print("=== BAND METADATA ===")
print(band0)

print("\n=== CRS ===")
print(band0.get("crs"))

print("\n=== CRS TRANSFORM ===")
print(band0.get("crs_transform") or band0.get("transform"))

print("\n=== FULL IMAGE INFO (truncated keys) ===")
print(list(info.keys()))

=== BAND METADATA ===
{'id': 'relaxed', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [61899, 72855], 'crs': 'PROJCS["Albers", \n  GEOGCS["WGS 84", \n    DATUM["WGS_1984", \n      SPHEROID["WGS 84", 6378140.0, 298.256999999996, AUTHORITY["EPSG","7030"]], \n      AUTHORITY["EPSG","6326"]], \n    PRIMEM["Greenwich", 0.0], \n    UNIT["degree", 0.017453292519943295], \n    AXIS["Longitude", EAST], \n    AXIS["Latitude", NORTH], \n    AUTHORITY["EPSG","4326"]], \n  PROJECTION["Albers_Conic_Equal_Area"], \n  PARAMETER["central_meridian", -96.0], \n  PARAMETER["latitude_of_origin", 23.0], \n  PARAMETER["standard_parallel_1", 29.5], \n  PARAMETER["false_easting", 0.0], \n  PARAMETER["false_northing", 0.0], \n  PARAMETER["standard_parallel_2", 45.5], \n  UNIT["m", 1.0], \n  AXIS["Easting", EAST], \n  AXIS["Northing", NORTH]]', 'crs_transform': [30, 0, -2361585, 0, -30, 3177435]}

=== CRS ===
PROJCS["Albers", 
  GEOGCS["WGS 84", 
    DATUM["WGS_1984"

In [31]:
proj = img.select(0).projection()

print("CRS (server):", proj.crs().getInfo())
print("Transform (server):", proj.transform().getInfo())
print("Nominal scale:", proj.nominalScale().getInfo())

CRS (server): None
Transform (server): PARAM_MT["Affine", 
  PARAMETER["num_row", 3], 
  PARAMETER["num_col", 3], 
  PARAMETER["elt_0_0", 30.0], 
  PARAMETER["elt_0_2", -2361585.0], 
  PARAMETER["elt_1_1", -30.0], 
  PARAMETER["elt_1_2", 3177435.0]]
Nominal scale: 30


In [29]:
import ee


SRC_COL = "projects/ee-tymc5571-cbi-module/assets/forest_masks_og"
DST_COL = "projects/ee-tymc5571-cbi-module/assets/forest_masks"
RAP_COL = "projects/rap-data-365417/assets/vegetation-cover-v3"

YEARS_MODE = "union"

SRC_BANDS = ["relaxed", "conservative"]
RAP_BAND = "TRE"
NEW_BAND = "rap10"


def get_years(ic):
    years = ic.aggregate_array("year").getInfo()
    years = [int(y) for y in years if y is not None]
    return sorted(set(years))


def make_masked_placeholder(projection):
    return (
        ee.Image.constant([0, 0])
        .rename(SRC_BANDS)
        .updateMask(ee.Image.constant([0, 0]))
        .setDefaultProjection(projection)
        .toByte()
    )


def build_rap10_for_year(year, target_projection):
    rap_year = (
        ee.ImageCollection(RAP_COL)
        .filter(ee.Filter.eq("year", year))
        .first()
    )

    rap10 = ee.Image(
        ee.Algorithms.If(
            rap_year,
            ee.Image(rap_year)
            .select(RAP_BAND)
            .gte(10)
            .unmask(0)
            .rename(NEW_BAND)
            .toByte()
            .setDefaultProjection(target_projection),
            ee.Image.constant(0)
            .rename(NEW_BAND)
            .toByte()
            .setDefaultProjection(target_projection)
        )
    )

    return rap10


def build_output_image(year, src_ic, reference_projection):
    src_year = src_ic.filter(ee.Filter.eq("year", year)).first()

    base_img = ee.Image(
        ee.Algorithms.If(
            src_year,
            ee.Image(src_year).select(SRC_BANDS),
            make_masked_placeholder(reference_projection)
        )
    )

    data_used = ee.String(
        ee.Algorithms.If(
            src_year,
            ee.Algorithms.If(
                ee.Image(src_year).propertyNames().contains("data_used"),
                ee.Image(src_year).get("data_used"),
                ""
            ),
            ""
        )
    )

    rap10 = build_rap10_for_year(year, reference_projection)

    out = (
        base_img
        .addBands(rap10)
        .set("year", year)
        .set("data_used", data_used)
    )

    return out


def safe_description(text):
    return text.replace("/", "_").replace("-", "_").replace(" ", "_")[:100]


def main():
    src_ic = ee.ImageCollection(SRC_COL)
    rap_ic = ee.ImageCollection(RAP_COL)

    src_years = get_years(src_ic)
    rap_years = get_years(rap_ic)

    if YEARS_MODE == "rap_only":
        years = rap_years
    else:
        years = sorted(set(src_years) | set(rap_years))

    if len(src_years) == 0:
        raise ValueError("Source forest mask collection has no images.")

    ref_img = ee.Image(src_ic.sort("year").first()).select("relaxed")
    ref_proj = ref_img.projection()
    ref_geom = ref_img.geometry()

    ref_info = ref_img.getInfo()
    band0 = ref_info["bands"][0]

    ref_crs = band0["crs"]
    ref_transform = band0["crs_transform"]

    print("Source years:", src_years)
    print("RAP years:", rap_years)
    print("Export years:", years)
    print("Reference band metadata:", band0)
    print("Reference CRS:", ref_crs)
    print("Reference transform:", ref_transform)

    tasks = []

    for year in years:
        out_img = build_output_image(year, src_ic, ref_proj)

        asset_id = f"{DST_COL}/year_{year}"
        description = safe_description(f"forest_masks_year_{year}")

        task = ee.batch.Export.image.toAsset(
            image=out_img,
            description=description,
            assetId=asset_id,
            region=ref_geom,
            crs=ref_crs,
            crsTransform=ref_transform,
            maxPixels=1e13
        )
        task.start()
        tasks.append((year, task))
        print(f"Started export for year {year}: {asset_id}")

    print(f"\nStarted {len(tasks)} export tasks.")

if __name__ == "__main__":
    main()

Source years: [1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
RAP years: [1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Export years: [1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Reference band metadata: {'id': 'relaxed', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [61899, 72855], 'crs': 'PROJCS["Albers", \n  GEOGCS["WGS 84", \n    DATUM["WGS_1984", \n      SPH

# Add forest masked layers and re-export WUMI

In [ ]:
import ee
import re

ee.Initialize(project="ee-tymc5571-cbi-module")

SRC_COLLECTION = "projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev_4326"
DST_COLLECTION = "projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev"
FOREST_MASKS_COLLECTION = "projects/ee-tymc5571-cbi-module/assets/forest_masks"

TARGET_CRS = "EPSG:5070"
TARGET_TRANSFORM = [30, 0, -2361600, 0, -30, 3177450]

MAX_PIXELS = 1e13
SHARD_SIZE = 512

START_TASKS = True
TEST_1 = False

FOREST_MASKS_COLLECTION = "projects/ee-tymc5571-cbi-module/assets/forest_masks"


def list_images(collection_id):
    result = ee.data.listImages(collection_id)
    return [img["name"] for img in result.get("images", [])]


def basename(asset_id):
    return asset_id.rstrip("/").split("/")[-1]


def safe_description(name):
    return re.sub(r"[^A-Za-z0-9_-]", "_", name)[:100]


def asset_exists(asset_id):
    try:
        ee.data.getAsset(asset_id)
        return True
    except Exception:
        return False


def get_image_year(image):
    year = image.get("year").getInfo()
    if year is None:
        raise ValueError("Source image is missing 'year' property.")
    return int(year)


def get_prior_forest_mask_image(year):
    prior_year = year - 1

    mask_col = (
        ee.ImageCollection(FOREST_MASKS_COLLECTION)
        .filter(ee.Filter.eq("year", prior_year))
    )

    n = mask_col.size().getInfo()

    if n == 0:
        return None, prior_year

    if n > 1:
        raise ValueError(
            f"Multiple forest_masks images found for prior year {prior_year}."
        )

    return ee.Image(mask_col.first()), prior_year


def get_band_names(image):
    return image.bandNames().getInfo()


def make_empty_masked_band(template_band, out_name):
    return (
        template_band
        .updateMask(ee.Image.constant(0))
        .rename(out_name)
    )


def make_masked_band(cbi, forest_mask, forest_band_name, out_name, available_mask_bands):
    if forest_band_name in available_mask_bands:
        return (
            cbi.updateMask(forest_mask.select(forest_band_name).gt(0))
            .rename(out_name)
        )
    else:
        return make_empty_masked_band(cbi, out_name)


def add_masked_bands(image):
    year = get_image_year(image)
    forest_mask, prior_year = get_prior_forest_mask_image(year)

    if forest_mask is None:
        cbi = image.select("CBI_bc")

        empty_relaxed = make_empty_masked_band(cbi, "CBI_bc_masked_relaxed")
        empty_conservative = make_empty_masked_band(cbi, "CBI_bc_masked_conservative")
        empty_rap10 = make_empty_masked_band(cbi, "CBI_bc_masked_rap10")

        out = image.addBands([
            empty_relaxed,
            empty_conservative,
            empty_rap10
        ])

        return out, prior_year

    cbi = image.select("CBI_bc")
    available_mask_bands = get_band_names(forest_mask)

    cbi_masked_relaxed = make_masked_band(
        cbi=cbi,
        forest_mask=forest_mask,
        forest_band_name="relaxed",
        out_name="CBI_bc_masked_relaxed",
        available_mask_bands=available_mask_bands
    )

    cbi_masked_conservative = make_masked_band(
        cbi=cbi,
        forest_mask=forest_mask,
        forest_band_name="conservative",
        out_name="CBI_bc_masked_conservative",
        available_mask_bands=available_mask_bands
    )

    cbi_masked_rap10 = make_masked_band(
        cbi=cbi,
        forest_mask=forest_mask,
        forest_band_name="rap10",
        out_name="CBI_bc_masked_rap10",
        available_mask_bands=available_mask_bands
    )

    out = image.addBands([
        cbi_masked_relaxed,
        cbi_masked_conservative,
        cbi_masked_rap10
    ])

    return out, prior_year


def launch_export(image_id):
    name = basename(image_id)
    dst_id = f"{DST_COLLECTION}/{name}"

    if asset_exists(dst_id):
        print(f"Skipping existing asset: {dst_id}")
        return None

    image = ee.Image(image_id)
    year = get_image_year(image)

    export_image, prior_year = add_masked_bands(image)


    task = ee.batch.Export.image.toAsset(
        image=export_image,
        description=safe_description(f"to5070_masked_{name}"),
        assetId=dst_id,
        region=image.geometry(),
        crs=TARGET_CRS,
        crsTransform=TARGET_TRANSFORM,
        maxPixels=MAX_PIXELS,
        shardSize=SHARD_SIZE
    )

    if START_TASKS:
        task.start()

    print(
        f"{'Started' if START_TASKS else 'Prepared'}: {dst_id} "
        f"(source year={year}, forest mask year={prior_year})"
    )

    return task


def main():
    image_ids = sorted(list_images(SRC_COLLECTION))
    print(f"Found {len(image_ids)} images")

    if TEST_1:
        image_ids = image_ids[:1]
        print("TEST_1=True -> running only first image")

    tasks = []
    for image_id in image_ids:
        task = launch_export(image_id)
        if task is not None:
            tasks.append(task)

    print(f"\nTasks created: {len(tasks)}")


if __name__ == "__main__":
    main()

Found 40 images
Started: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev/WUMI2024a_allsev_1985 (source year=1985, forest mask year=1984)
Started: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev/WUMI2024a_allsev_1986 (source year=1986, forest mask year=1985)
Started: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev/WUMI2024a_allsev_1987 (source year=1987, forest mask year=1986)
Started: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev/WUMI2024a_allsev_1988 (source year=1988, forest mask year=1987)
Started: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev/WUMI2024a_allsev_1989 (source year=1989, forest mask year=1988)
Started: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev/WUMI2024a_allsev_1990 (source year=1990, forest mask year=1989)
Started: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev/WUMI2024a_allsev_1991 (source year=1991, forest mask year=1990)
Started: projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev/WUMI2024a_allsev